# 02 – Vorverarbeitung 

Hier werden die Beschwerdetexte der Stichprobe in Wortlisten umgewandelt, die später vektorisiert und ausgewertet werden.

**Eingabe:** 
Stichprobe von 20.000 Beschwerden, gezogen aus den 366.945 Beschwerden mit Text ohne Duplikate

**Ausgabe:** 
data/tokens_final.pkl mit 19.807 Beschwerden.

**Schritte:**
1. Platzhalter wie „XXXX" und Datumsmuster mit regulären Ausdrücken entfernen
2. Mit spaCy (`en_core_web_sm`) in Wörter zerlegen und auf Grundformen bringen
3. Filtern: nur Nomen, Verben und Adjektive ohne Stoppwörter, mindestens drei Buchstaben
4. Fehlerhafte Grundformen und häufige Tippfehler mit einer Korrektur-Tabelle beheben
5. Sehr seltene Wörter entfernen (Untergrenze `min_df`)
6. Zu kurze Beschwerden aussortieren
7. Ergebnis speichern

In [ ]:
import re
import pandas as pd
import spacy
from collections import Counter
import time
import random
from pathlib import Path

In [ ]:
nlp = spacy.load("en_core_web_sm")

# Regeln
ALLOWED_POS = {"NOUN", "VERB", "ADJ"}
STOPWORDS = set(nlp.Defaults.stop_words)
MIN_LENGTH = 3  # mindestens 3 Buchstaben
LEMMA_FIXES = {
    "taxis": "tax",
    "taxes": "tax",
    "pende": "pending",
    "datum": "data",
    "monie": "money",
    "reinserte": "reinsert",
    "hop": "hope",
    "recieve": "receive",  # sehr häufiger Tippfehler
}
# Wort muss in min 10 verschiedenen Beschwerden vorkommen
MIN_DF = 10   
# zu kurze Texte werden aussortiert
MIN_TOKENS = 4   # Einschränkung niedrig angesetzt um möglichst wenig semantischen Verlust zu haben



In [ ]:
path = Path("data/sample_20000.csv")
if not path.exists():
    path = Path("sample/sample_500.csv") 

sample = pd.read_csv(path)
col = "Consumer complaint narrative"
print("Eingabe:", path, sample.shape)

In [ ]:
def clean_text(text):
    text = re.sub(r"X{2,}(/X{2,})*(/\d{2,4})?", " ", text)  # XXXX, XX/XX/2018
    text = re.sub(r"\{\$[\d.,]+\}", " ", text)  # Beträge wie {$100.00}
    text = re.sub(r"\s+", " ", text).strip()  # mehrfache Leerzeichen
    return text


beispiel = sample[col].iloc[0]
print(beispiel[:300])
print("---")
print(clean_text(beispiel)[:300])

In [ ]:
sample["text_clean"] = sample[col].apply(clean_text)

print("Texte mit XX vorher:", sample[col].str.contains("XX").sum())
print("Texte mit XX nachher:", sample["text_clean"].str.contains("XX").sum())

In [ ]:

doc = nlp(sample["text_clean"].iloc[0][:300])

for token in doc:
    print(
        f"{token.text:15} {token.lemma_:15} {token.pos_:8} Stoppwort: {token.is_stop}"
    )

In [ ]:

# erste Fassung ohne Korrektur-Tabelle für Testlauf
def extract_tokens_test(doc):
    return [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and token.pos_ in ALLOWED_POS
        and len(token.lemma_) > 2
    ]

# Beispielausgabe zur Kontrolle
print(extract_tokens_test(doc))

## Filterregeln 

**Ablauf:** 
Zuerst entferne ich Platzhalter mit Regex. Danach zerlegt spaCy die Texte in Wörter und bildet den Stamm. Erst am Ende schreibe ich  klein, weil spaCy Wortarten am Originaltext zuverlässiger erkennt.

**Behalten werden** 
Wörter, die
- nur aus Buchstaben bestehen,
- keine Stoppwörter sind,
- Nomen, Verben oder Adjektive sind,
- mehr als zwei Buchstaben haben.

**Begründung**
- **Eigennamen verworfen:** Eigennamen sind in der Regel irrelevant und erhöhen unnötiges Rauschen
- **Adverbien verworfen:** Wörter wie „currently" tragen wenig zum Thema bei.
- **Verben behalten:** Sie tragen zur Semantik bei
- **Mindestlänge über zwei Buchstaben:** So fallen Kürzel weg, aber kurze Wörter wie „car" oder „bus" bleiben erhalten.



In [ ]:
test = sample["text_clean"].head(1000)

docs = list(nlp.pipe(test, disable=["parser", "ner"], batch_size=100))
tokens_test = [extract_tokens_test(d) for d in docs]

print(len(tokens_test))
print(tokens_test[0])
print(tokens_test[1])

## Auffälligkeit: Lemmatisierungsfehler bei „taxes"

**Problem:** 
Das Modell `en_core_web_sm` bildet aus „taxes" in 63 von 64 Fällen die Grundform „taxis" statt „tax". Als Ursache vermute ich, dass das Modell die Endung wie bei „crises → crisis" behandelt.

**Prüfung:**
 Im Testlauf habe ich die 200 häufigsten Paare aus Originalwort und Grundform durchgelesen. Dort ist nur „taxes → taxis" aufgefallen. Im Lauf über alle 20.000 Texte kamen weitere Fehler zum Vorschein (siehe unten).

**Entscheidung:** 
Ich bleibe bei `en_core_web_sm` und korrigiere nachträglich mit einer Tabelle (`LEMMA_FIXES`). Meine Regel war: Bei einem Einzelfall korrigiere ich, bei mehreren häufigen Fehlern teste ich zuerst `en_core_web_md`. Später kamen mehrere Fehler dazu, ``en_core_web_md`` habe ich nicht getestet, weil die Ausreisser gering sind und sich mit der Tabelle beheben lassen.



In [ ]:
# wie oft kommt "taxis" und "tax" vor
counter = Counter(word for tokens in tokens_test for word in tokens)
print("taxis:", counter["taxis"], "| tax:", counter["tax"])

In [ ]:
# wird "taxes" immer falsch lemmatisiert
taxes_lemmas = Counter(
    t.lemma_.lower() for d in docs for t in d if t.text.lower() == "taxes"
)
print(taxes_lemmas)

In [ ]:
pairs = Counter(
    (t.text.lower(), t.lemma_.lower())
    for d in docs
    for t in d
    if t.is_alpha and not t.is_stop and t.pos_ in ALLOWED_POS and len(t.lemma_) > 2
)

changed = [(w, l, c) for (w, l), c in pairs.items() if w != l]
changed.sort(key=lambda x: -x[2])

for w, l, c in changed[:60]:
    print(f"{w:20} -> {l:20} {c}")

In [ ]:
for w, l, c in changed[60:200]:
    print(f"{w:20} -> {l:20} {c}")

## Verarbeitung aller 20.000 Texte

Die festgelegten Filterregeln werden auf die gesamte Stichprobe angewendet. Zusätzlich greift eine Korrektur-Tabelle (`LEMMA_FIXES`) für fehlerhafte Grundformen und  Tippfehler. Stoppwörter werden auch auf der Grundform geprüft, weil token.is_stop nur die Wortform prüft und dadurch Verben wie „get“ oder „give“ erst auf Lemma-Ebene herausfallen.

**Laufzeit:** 
252,9 s für 20.000 Texte (ca. 4 Minuten, 12,6 ms pro Text).

**Kontrolle nach dem Lauf:**
- Die acht fehlerhaften Formen kommen im Ergebnis nicht mehr vor.
- Die korrigierten Wörter treten in den erwarteten Häufigkeiten auf

**Bewusst nicht angepasst:**
- „report" und „reporting" bleiben ohne Sonderregel, um die Zahl der Einzelfallkorrekturen klein zu halten.
- Mehrdeutige Grundformen („state", „lie") bleiben bestehen und fallen kaum ins Gewicht.

In [ ]:


# Filtert relevante Grundformen aus dem Doc und wendet Regeln an
def extract_tokens(doc, pair_counts=None):
    tokens = []
    for token in doc:
        if not token.is_alpha or token.is_stop or token.pos_ not in ALLOWED_POS:
            continue

        raw_lemma = token.lemma_.lower()
        lemma = LEMMA_FIXES.get(raw_lemma, raw_lemma)

        if len(lemma) < MIN_LENGTH or lemma in STOPWORDS:
            continue

        if pair_counts is not None and token.text.lower() != raw_lemma:
            pair_counts[(token.text.lower(), raw_lemma)] += 1

        tokens.append(lemma)
    return tokens

In [ ]:
texts = sample["text_clean"]
pair_counts = Counter()  # bei jedem Lauf wieder bei 0 starten
tokens_all = []

start = time.perf_counter()
for d in nlp.pipe(texts, disable=["parser", "ner"], batch_size=100):
    tokens_all.append(extract_tokens(d, pair_counts))
dauer = time.perf_counter() - start

sample["tokens"] = tokens_all
print(
    f"{len(texts)} Texte in {dauer:.1f} s ({dauer / len(texts) * 1000:.1f} ms pro Text)"
)

In [ ]:
all_tokens = [tok for toks in sample["tokens"] for tok in toks]
token_counts = Counter(all_tokens)

print("Falsche Grundformen (sollten 0 sein):")
for wrong in LEMMA_FIXES:
    print(f"  {wrong:10} {token_counts[wrong]}")

print("Korrigierte Grundformen:")
for right in sorted(set(LEMMA_FIXES.values())):
    print(f"  {right:10} {token_counts[right]}")

print(
    "call:",
    token_counts["call"],
    "| report:",
    token_counts["report"],
    "| reporting:",
    token_counts["reporting"],
)

In [ ]:
#  häufigste Paare
for (w, l), c in pair_counts.most_common(300):
    print(f"{w:20} -> {l:20} {c}")

In [ ]:
#  häufigste Wörter
Counter(all_tokens).most_common(100)

In [ ]:
kandidaten = [
    "call",
    "say",
    "get",
    "take",
    "give",
    "tell",
    "receive",
    "send",
    "ask",
    "pay",
]
print("Stoppwörter:", sorted(w for w in kandidaten if w in STOPWORDS))
print("Keine Stoppwörter:", sorted(w for w in kandidaten if w not in STOPWORDS))

In [ ]:
# Wörter 300-600
for (w, l), c in pair_counts.most_common()[300:600]:
    print(f"{w:20} -> {l:20} {c}")

## Vokabular-Filter 

Um Tippfehler und extrem seltene Begriffe auszusortieren, werden nur Wörter behalten, die in mindestens 10 Beschwerden vorkommen (`MIN_DF = 10`). 

Auf eine Obergrenze wird verzichtet, da selbst sehr häufige Wörter wie „credit“ oder „account“ zentrale Fachbegriffe der Beschwerden sind und nicht entfernt werden sollen.

In [ ]:
#checke sehr seltene und sehr häufige Wörter
doc_freq = Counter(w for toks in sample["tokens"] for w in set(toks))
n_docs = len(sample)

for w, c in doc_freq.most_common(60):
    print(f"{w:15} {c:6} {c / n_docs:6.1%}")

In [ ]:
#Abfrage für Untergrenze
print("Verschiedene Wörter insgesamt:", len(doc_freq))

for min_df in [2, 5, 10, 20, 50]:
    n = sum(1 for c in doc_freq.values() if c >= min_df)
    print(f"min_df={min_df:3}: {n} Wörter bleiben")

In [ ]:
total = sum(len(t) for t in sample["tokens"])

for min_df in [2, 5, 10, 20]:
    v = {w for w, c in doc_freq.items() if c >= min_df}
    kept = sum(1 for toks in sample["tokens"] for t in toks if t in v)
    print(f"min_df={min_df:3}: {kept / total:.1%} aller Wörter bleiben")

In [ ]:

random.seed(42)

for lo, hi in [(2, 5), (5, 10)]:
    wörter = [w for w, c in doc_freq.items() if lo <= c < hi]
    print(f"{lo} bis {hi - 1} Beschwerden ({len(wörter)} Wörter):")
    print(random.sample(wörter, min(40, len(wörter))), "\n")

In [ ]:
vocab = {w for w, c in doc_freq.items() if c >= MIN_DF}
sample["tokens_filtered"] = sample["tokens"].apply(lambda toks: [t for t in toks if t in vocab])

sample["n_tokens"] = sample["tokens_filtered"].apply(len)
print(sample["n_tokens"].describe())


## Zu kurze Beschwerden aussortieren

Nach den bisherigen Filtern bleiben bei manchen Beschwerden nur sehr wenige Wörter übrig, die kaum noch ein Thema tragen. Deshalb werden Beschwerden mit weniger als 4 Wörtern ausgeschlossen.

Dabei entfallen 193 Beschwerden (ca 1 %), sodass 19.807 Beschwerden erhalten bleiben. Die Datenqualität ändert sich dadurch praktisch nicht.

In [ ]:
#Finde heraus was eine sinnvolle Untergrenze ist
for grenze in [1, 3, 5, 8, 10, 15, 20]:
    n = (sample["n_tokens"] < grenze).sum()
    print(f"weniger als {grenze:2} Wörter: {n:4} Beschwerden ({n / len(sample):.1%})")

In [ ]:
kurz_alle = sample[(sample["n_tokens"] >= 3) & (sample["n_tokens"] < 10)]
kurz = kurz_alle.sample(min(12, len(kurz_alle)), random_state=42)
for _, row in kurz.iterrows():
    print(row["n_tokens"], "|", row["text_clean"][:150])
    print("   ", row["tokens_filtered"], "\n")

In [ ]:
sample_final = sample[sample["n_tokens"] >= MIN_TOKENS].copy()
print(len(sample), "->", len(sample_final), f"({len(sample) - len(sample_final)} entfernt)")

vergleich = pd.DataFrame({
    "vorher": sample["Product"].value_counts(normalize=True),
    "nachher": sample_final["Product"].value_counts(normalize=True),
})
vergleich["Differenz"] = vergleich["nachher"] - vergleich["vorher"]
print((vergleich * 100).round(1).sort_values("vorher", ascending=False))

In [ ]:
print(sample_final["n_tokens"].describe())
print(sample_final["tokens_filtered"].iloc[0])

In [ ]:
final_df = Counter(w for toks in sample_final["tokens_filtered"] for w in set(toks))
unter = {w for w, c in final_df.items() if c < MIN_DF}

total = sample_final["n_tokens"].sum()
betroffen = sum(1 for toks in sample_final["tokens_filtered"] for t in toks if t in unter)

print(f"{len(unter)} von {len(final_df)} Wörtern kommen in weniger als {MIN_DF} Beschwerden vor")
print(f"Das sind {betroffen} von {total} Wortvorkommen ({betroffen / total:.2%})")

## Ergebnis der Vorverarbeitung

Nach allen Schritten bleiben 19.807 Beschwerden mit einem Vokabular von 3.571 Wörtern (insgesamt rund 1,24 Mio. Wörter) übrig.

Das Ergebnis wird als `data/tokens_final.pkl` gespeichert, damit die Wortlisten direkt als Python-Listen für die Vektorisierung bereitstehen (die Datei selbst liegt nicht im Repository).

In [ ]:
# Zusammenfassung der Vorverarbeitung
vokabular = {w for toks in sample_final["tokens_filtered"] for w in toks}

print("Beschwerden ursprünglich:", len(sample))
print("Beschwerden final:", len(sample_final))
print("Entfernte Beschwerden:", len(sample) - len(sample_final))
print("Vokabular:", len(vokabular))
print("Wörter insgesamt:", sample_final["n_tokens"].sum())
print("Median Wörter pro Beschwerde:", sample_final["n_tokens"].median())
print(f"Mittelwert Wörter pro Beschwerde: {sample_final['n_tokens'].mean():.1f}")

In [ ]:
spalten = ["Complaint ID", "Product", "Issue", "text_clean", "tokens_filtered", "n_tokens"]
Path("data").mkdir(exist_ok=True)
sample_final[spalten].to_pickle("data/tokens_final.pkl")

check = pd.read_pickle("data/tokens_final.pkl")
print(check.shape)
print(type(check["tokens_filtered"].iloc[0]))